In [14]:
import paho.mqtt.client as mqtt
import pandas as pd
import re
import time
from datetime import datetime
import os

# ─── Configuration ─────────────────────────────────────────────
BROKER_ADDRESS = "127.0.0.1"
MQTT_TOPIC = "imu/data"
RECORD_SECONDS = 2        # duration per session
NUM_WALKING = 3           # number of walking sessions
NUM_FALLING = 3           # number of falling sessions
BASE_OUTPUT_DIR = "data"  # base folder for all tests

os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

# ─── Function to Record One Session ───────────────────────────
def record_session(label, session_number, test_id, test_folder):
    data_points = []
    collecting = True

    def on_connect(client, userdata, flags, rc):
        if rc == 0:
            client.subscribe(MQTT_TOPIC)
        else:
            print("❌ Connection failed:", rc)

    def on_message(client, userdata, msg):
        nonlocal data_points, collecting
        if not collecting:
            return
        payload = msg.payload.decode("utf-8").strip()
        matches = re.findall(r"\[([0-9\.,\- ]+)\]", payload)
        for m in matches:
            vals = [float(x) for x in m.split(",")]
            data_points.append(vals)
        print(f"📡 {label} {session_number}: {len(data_points)} samples", end="\r")

    client = mqtt.Client()
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect(BROKER_ADDRESS, 1883, 60)

    print(f"\n▶️ Recording {label.upper()} session {session_number} ({RECORD_SECONDS}s)...")
    client.loop_start()
    start_time = time.time()


    while time.time() - start_time < RECORD_SECONDS:
        time.sleep(0.05)

    collecting = False
    client.loop_stop()
    client.disconnect()

    if len(data_points) == 0:
        print(f"⚠️ No data received for {label} {session_number}")
        return

    # ─── Create DataFrame ───────────────────────────────────────
    df = pd.DataFrame(data_points, columns=["t", "acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"])
    df["dt"] = df["t"].diff()
    df.loc[df["dt"] < 0, "dt"] += 2**32
    df = df.dropna().reset_index(drop=True)
    df["dt_sec"] = df["dt"] / 1_000_000.0

    avg_freq = 1 / df["dt_sec"].mean()
    jitter_ms = df["dt_sec"].std() * 1000

    date_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"test_{test_id}_{label}_{session_number}_{len(df)}samples_{RECORD_SECONDS}s.csv"
    filepath = os.path.join(test_folder, filename)
    df.to_csv(filepath, index=False)

    print(f"\n✅ Saved {label} session {session_number} to {filepath}")
    print(f"   Avg freq: {avg_freq:.2f} Hz | Jitter: {jitter_ms:.3f} ms")

# ─── Validate Test ID ──────────────────────────────────────────
def get_valid_test_id():
    while True:
        test_id = input("Enter test number (numbers only, e.g. 01): ").strip()
        if not test_id:
            print("⚠️ Test number cannot be empty. Try again.")
            continue
        if not test_id.isdigit():
            print("⚠️ Invalid test number. Use only digits (no letters or symbols).")
            continue
        return test_id

# ─── Main Collection ──────────────────────────────────────────
test_id = get_valid_test_id()

date_str = datetime.now().strftime("%Y%m%d")
test_folder = os.path.join(BASE_OUTPUT_DIR, f"test_{test_id}_{date_str}")
os.makedirs(test_folder, exist_ok=True)

print(f"\n📁 Created folder for this test: {test_folder}")

# Walking sessions
print(f"\n🚶 Starting WALKING sessions for Test {test_id}")
for i in range(1, NUM_WALKING + 1):
    input(f"\nPress ENTER when ready for WALKING session {i}/{NUM_WALKING}...")
    record_session("walking", i, test_id, test_folder)
    time.sleep(2)

# Falling sessions
print(f"\n💥 Starting FALLING sessions for Test {test_id}")
for i in range(1, NUM_FALLING + 1):
    input(f"\nPress ENTER when ready for FALLING session {i}/{NUM_FALLING}...")
    record_session("falling", i, test_id, test_folder)
    time.sleep(2)

print(f"\n🎉 All walking and falling sessions completed for Test {test_id}!")
print(f"📦 Results saved in: {test_folder}")



📁 Created folder for this test: data/test_3_20251023

🚶 Starting WALKING sessions for Test 3


/var/folders/m1/qxpcclg55hg_8gcmt4vpkrq80000gn/T/ipykernel_92979/792100012.py:40: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client()



▶️ Recording WALKING session 1 (2s)...
⚠️ No data received for walking 1

▶️ Recording WALKING session 2 (2s)...
⚠️ No data received for walking 2

▶️ Recording WALKING session 3 (2s)...
⚠️ No data received for walking 3

💥 Starting FALLING sessions for Test 3

▶️ Recording FALLING session 1 (2s)...
⚠️ No data received for falling 1

▶️ Recording FALLING session 2 (2s)...
⚠️ No data received for falling 2

▶️ Recording FALLING session 3 (2s)...
⚠️ No data received for falling 3

🎉 All walking and falling sessions completed for Test 3!
📦 Results saved in: data/test_3_20251023
